In [3]:
# ============================================================================
# NanoString RCC -> one master CSV: identity + demographics + raw counts
#
# Run top to bottom, once. Split on the "# === CELL n ===" lines if you want
# it as notebook cells.
#
# Output is a single CSV, one row per RCC file. What used to be separate
# tabs are now filter columns: record_type, specimen_type, needs_review,
# is_non_blood, qc_pass. Nothing is ever dropped.
#
# CSV ONLY: no Excel files are written and openpyxl/xlsxwriter are not needed.
#
# Demographics come from MetadataPatients_NoPHI, joined on Enrollment ID and
# Alternate Enrollment ID. Birth date is read only to compute age at the
# sample date and is then discarded, so no DOB reaches any output.
#
# Sample dates: only the date in the RCC ID field counts as the collection
# date. run_date (instrument header, then filename) is kept separately.
# ============================================================================


# === CELL 1 : configuration =================================================
import os, re, glob, shutil, hashlib, warnings
from datetime import datetime

import numpy as np
import pandas as pd
%pip install -q pyarrow
import importlib; importlib.invalidate_caches()

# ---- paths -----------------------------------------------------------------
FOLDER = "/home/jovyan/rcc_extracted/Nanostring_RCC"   # RCC files + metadata
OUT    = "/mnt/filesystems/nanostring_out"                 # everything is written here

# metadata / demographics file; .csv / .xlsx / .xls all work as INPUT.
# This is the single source of demographics - there is no separate sheet.
# Matching ignores capitalisation and punctuation. The RCC folder, its parent,
# and every path in METADATA_SEARCH_DIRS are searched, so the metadata does
# NOT have to sit next to the RCC files - it can stay at the mount point.
METADATA_GLOB = "MetadataPatients_NoPHI*"
METADATA_SEARCH_DIRS = ["/mnt/session_data", "/home/jovyan", "/mnt"]

# Extra patients to add on top of the main metadata. List full paths to .xlsx /
# .csv files whose headers look like the main sheet (Enrollment ID, Sex, ...).
# Column names are matched loosely; missing columns are left blank. If the same
# enrollment ID is in both, the MAIN sheet wins. Example:
#   METADATA_EXTRA = ["/home/jovyan/rcc_extracted/extra_patients.xlsx"]
METADATA_EXTRA = []

# ---- identifiers -----------------------------------------------------------
# Birth date is read (age cannot be computed without it) but never written to
# any output. Only the derived age survives. Set False only if you specifically
# need DOB in the output and your environment is approved for it.
DROP_BIRTH_DATE = True

# Initials are also an identifier. They appear in the original RCC filenames
# and are useful for eyeballing matches, so they are kept by default.
DROP_INITIALS = False

# ---- what to do with the RCC files themselves ------------------------------
#   "map_only"  -> write the old->new lookup, touch nothing on disk  (default)
#   "copy_flat" -> copy renamed files into OUT/renamed/
#   "copy_tree" -> copy into OUT/renamed/<diagnosis>/<patient_id>/
RENAME_MODE = "map_only"

# Put EVERY probe's raw count (endogenous genes, housekeeping, positive and
# negative controls, ~90 columns) into the master CSV, not just the 36 panel
# genes. Raw counts either way - nothing is normalised.
INCLUDE_ALL_PROBES = True

# which pieces go into the new filename (patient ID + date are always included)
INCLUDE_INITIALS = True    # already present in the original names; aids eyeballing
INCLUDE_DX       = True    # diagnosis slug
INCLUDE_VISIT    = True    # v01, v02 ... ordered by sample date within patient
INCLUDE_SPECIMEN = True    # only appended when it is NOT whole blood
INCLUDE_LANE     = True    # lane number, keeps multiplexed runs distinct

# ---- panel -----------------------------------------------------------------
TARGET_GENES = [
    "CXCL10","DDX60","EPSTI1","GBP1","HERC5","HERC6","IFI27","IFI44","IFI44L",
    "IFI6","IFIT1","IFIT2","IFIT3","IFIT5","IFNA2","IFNB1","IRF7","ISG15",
    "LAMP3","LY6E","MX1","OAS1","OAS2","OAS3","OASL","RSAD2","RTP4","SAMD9",
    "SIGLEC1","SOCS1","SPATS2L","USP18",
]
HOUSEKEEPING = ["ALAS1","HPRT1","TBP","TUBB"]

# ---- QC thresholds (NanoString defaults) -----------------------------------
BD_MIN, BD_MAX = 0.05, 2.25   # binding density
FOV_MIN        = 0.75         # fraction of fields of view successfully counted

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)
os.makedirs(OUT, exist_ok=True)

# === CELL 1b : locate the RCC files (replaces trusting FOLDER blindly) ======
import zipfile

SEARCH_ROOTS  = ["/home/jovyan", "/mnt/session_data", "/mnt"]
MAX_DEPTH     = 5
EXTRACT_TO    = "/home/jovyan/rcc_extracted"     # used only if the files are zipped

def _scan(roots, max_depth=MAX_DEPTH):
    """-> ({dir: n_rcc}, [zip paths]). Skips hidden dirs and python packages."""
    found, zips, seen = {}, [], set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip(os.sep).count(os.sep)
        for dp, dn, fn in os.walk(root):
            rp = os.path.realpath(dp)
            if rp in seen:
                dn[:] = []; continue
            seen.add(rp)
            if dp.count(os.sep) - base >= max_depth:
                dn[:] = []
            dn[:] = [d for d in dn if not d.startswith(".")
                     and d not in ("site-packages", "node_modules", "__pycache__")]
            n = sum(f.lower().endswith(".rcc") for f in fn)
            if n:
                found[dp] = n
            zips += [os.path.join(dp, f) for f in fn if f.lower().endswith(".zip")]
    return found, zips

def find_rcc_folder(hint):
    if os.path.isdir(hint) and any(f.lower().endswith(".rcc") for f in os.listdir(hint)):
        return hint
    print(f"{hint} missing or has no .RCC files - searching...")
    found, zips = _scan(SEARCH_ROOTS)

    if not found and zips:                       # maybe they are still zipped
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(m.lower().endswith(".rcc") for m in zf.namelist()):
                        print(f"extracting {z} -> {EXTRACT_TO}")
                        zf.extractall(EXTRACT_TO)
            except zipfile.BadZipFile:
                continue
        found, _ = _scan([EXTRACT_TO])

    if not found:
        raise FileNotFoundError(
            "No .RCC files or RCC-containing zips under "
            f"{SEARCH_ROOTS}. Zips seen: {zips[:10] or 'none'}. "
            "Set FOLDER in Cell 1 to the right place.")

    ranked = sorted(found.items(), key=lambda kv: -kv[1])
    for d, n in ranked[:5]:
        print(f"  {n:6d} RCC files  {d}")
    best = ranked[0][0]
    print(f"using {best}")
    return best

FOLDER = find_rcc_folder(FOLDER)
# === CELL 2 : vocabulary and ID helpers =====================================

def normalize_id(x):
    """O-6033 -> O6033, c6441 -> C6441, 1013.0 -> 1013, LD0195.0 -> LD0195."""
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""):
        return ""
    s = re.sub(r"\.0$", "", s)   # Excel float suffix
    return s.upper().replace("-", "").replace(" ", "").replace("_", "")

def slug(s, maxlen=18):
    """Free text -> a short uppercase token safe for filenames and tags."""
    s = re.sub(r"[^A-Za-z0-9]+", "-", str(s or "")).strip("-").upper()
    return s[:maxlen].strip("-")


def norm_colname(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())


# --- specimen: WHAT the sample is. Independent of WHOSE it is. --------------
# Order matters; first hit wins. Anything unmatched is assumed whole blood,
# which is what the overwhelming majority of these runs are.
SPECIMEN_PATTERNS = [
    ("PBMC",         r"\bpbmcs?\b"),
    ("monocyte",     r"monocyt|\bmonos?\b"),
    ("neutrophil",   r"neutroph?il?|\bpmns?\b"),
    ("CD4_Tcell",    r"\bcd4\b"),
    ("CD8_Tcell",    r"\bcd8\b"),
    ("NK_cell",      r"\bnk[\s_-]*cells?\b"),
    ("B_cell",       r"\bb[\s_-]*cells?\b"),
    ("T_cell",       r"\bt[\s_-]*cells?\b"),
    ("lymphocyte",   r"lymphocyt"),
    ("macrophage",   r"macrophages?|\bmdms?\b"),
    ("BAL",          r"\bbals?\b|bronchoalveolar"),
    ("lung_tissue",  r"\blung\b"),
    ("liver_tissue", r"bio\s*liv|\bliver\b"),
    ("fibroblast",   r"fibroblast|\bfibs?\b"),
    ("keratinocyte", r"keratinocyt"),
    ("bone_marrow",  r"bone\s*marrow"),
    ("skin_biopsy",  r"\bskin\b|biopsy"),
]
DEFAULT_SPECIMEN = "whole_blood"

# --- evidence patterns: WHAT KIND of record it is ---------------------------
STANDARD_PAT    = re.compile(r"(sd[\s_]*tube|sdtube|\bsd\d\b|\bsst\b|standard|\b\d+\s*ds\b)", re.I)
PLACEHOLDER_PAT = re.compile(r"\b(xyz|dummy|test)\b|\bsample\d+\b", re.I)
CONTROL_PAT     = re.compile(r"\bhc\d?\b|healthy|\bhk\d*\b|\brv\d|bio\s*liv\s*control|\bcontrol\b", re.I)
OTHER_COHORT_PAT= re.compile(r"HGR\d|PAX-|NAP\d|PIMS|NIH\d|STANF|-APAH|\bNH-?[CP]?\d|"
                             r"\bIDT[\s-]|KFSHRC|HERSEI|\bBS[\s-]?RG\b|BS-SPECIAL", re.I)
COVID_PAT       = re.compile(r"covid|\bcv\d|CV0\d", re.I)
STIM_PAT        = re.compile(r"\bTNF\b|\bLPS\b|\bpIC\b|\bKLA\b|\bIFN[ab]?\s*stim|"
                             r"\+\s*G\b|\bG\d\b|\bNT\b|\bWT\b|\bV\d{2,3}[A-Z]\b|SAVI|SAMD9?L?\s*mut|"
                             r"\b\d+\s*min\b", re.I)
TIMEPOINT_PAT   = re.compile(r"\b\d+\s*h(r|rs)?\b|\b\d+h\b|\bt0\b|\bt1\b|PREAM|PREPM|POSTAM|POSTPM|"
                             r"premorn|post-?infusion|\bBSL\b|\bbaseline\b|run\s*2", re.I)

# --- parsing patterns -------------------------------------------------------
MMDDYY        = re.compile(r"\b(\d{6})\b")
ID_IN_FIELD   = re.compile(r"\b([ROCP]-?\d{3,5}|O\d{4}|\d{2}-\d{3}[A-Z]?|\d{4})\b", re.I)
INITIALS_LEAD = re.compile(r"^(?:\d{2})?\s*([A-Za-z]{2,5}\d?)\+?[\s_-]", re.I)
OP_FROM_CART  = re.compile(r"([A-Za-z]{2,5})")

# --- manual override layer --------------------------------------------------
# These win over anything auto-derived from the metadata file. Add to them as
# you resolve rows flagged needs_review.
MANUAL_INITIALS_MAP = {k: normalize_id(v) for k, v in {
    "AMG":"1074","AMK":"1406","AMK2":"C-6450","AMKT":"1257","ANV":"1613",
    "ANV2":"R2093","AWT":"1385","CEG":"1166","CEG2":"1384","EDS":"1469",
    "FNU":"1457","HBS":"2052","IRB":"1013","IRB2":"R1049","JJK":"1048",
    "JMM":"2308","JMM2":"O-6155","JTD":"1565","KAC2":"C-6589","KMM":"C-6575",
    "KVR":"1188","LNM2":"2355","MAF":"2209","MES":"2342","MES2":"R2398",
    "MES3":"C-6434","MSV":"2276","MSW":"C-6567","NBM":"1588","NBM2":"1676",
    "NTJM":"1377","OPS":"2106","PEDS":"R1831","RNS":"R1780","RNS2":"R2100",
    "SNT":"1340","SNT2":"2077","SNT3":"R2078","SNT4":"R2396","TMS2":"C-6649",
    "TRAM":"1370","YNC2":"1754",
}.items()}

# (initials, wrong_id_as_written) -> correct_id. Only add after you have
# confirmed it against the enrollment database.
VERIFIED_TYPOS = {(k[0], normalize_id(k[1])): normalize_id(v) for k, v in {
    ("IRB", "1000"): "1013",
}.items()}

# initials you already know are ambiguous and should never be auto-filled
KNOWN_AMBIGUOUS = {"ANV","MAF","MES","RNS","EOS","KNM2","LHSC","VNC","XXX"}


# === CELL 3 : RCC file parser ===============================================
SECTION_RE = re.compile(r"<(\w+)>(.*?)</\1>", re.S)


def _kv_block(text):
    """Parse a 'Key,Value' block into a dict."""
    d = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or "," not in line:
            continue
        k, _, v = line.partition(",")
        k = k.strip()
        if k:
            d[k] = v.strip()
    return d


def parse_rcc(path):
    """Return (attrs dict, list of (code_class, gene, accession, count))."""
    with open(path, "r", errors="replace") as fh:
        txt = fh.read()

    sections = {m.group(1): m.group(2) for m in SECTION_RE.finditer(txt)}

    # Tolerate files with a missing closing tag: fall back to a forward scan.
    if "Code_Summary" not in sections and "<Code_Summary>" in txt:
        sections["Code_Summary"] = txt.split("<Code_Summary>", 1)[1]
    if not sections:
        sections = {"Sample_Attributes": txt[:3000]}

    hdr  = _kv_block(sections.get("Header", ""))
    samp = _kv_block(sections.get("Sample_Attributes", ""))
    lane = _kv_block(sections.get("Lane_Attributes", ""))

    counts = []
    for line in sections.get("Code_Summary", "").splitlines():
        line = line.strip()
        if not line or line.startswith("<"):
            continue
        parts = line.split(",")
        if len(parts) < 4:
            continue
        cc, nm, acc, ct = (p.strip() for p in parts[:4])
        if cc.lower() == "codeclass":
            continue
        try:
            ct = int(float(ct))
        except ValueError:
            continue
        counts.append((cc, nm, acc, ct))

    attrs = {
        "id_field_raw":   samp.get("ID", ""),
        "sample_owner":   samp.get("Owner", ""),
        "sample_comment": samp.get("Comments", ""),
        "run_date_raw":   samp.get("Date", ""),
        "gene_rlf":       samp.get("GeneRLF", "") or hdr.get("GeneRLF", ""),
        "lane_id":        lane.get("ID", ""),
        "fov_count":      lane.get("FovCount", ""),
        "fov_counted":    lane.get("FovCounted", ""),
        "binding_density":lane.get("BindingDensity", ""),
        "cartridge":      lane.get("CartridgeID", ""),
        "scanner_id":     lane.get("ScannerID", ""),
        "rcc_version":    hdr.get("FileVersion", ""),
        "n_probes":       len(counts),
    }
    return attrs, counts


def _geomean(vals):
    vals = [v for v in vals if v is not None and v > 0]
    if not vals:
        return np.nan
    return float(np.exp(np.mean(np.log(vals))))


def qc_from_counts(counts):
    """Standard NanoString QC. Not an IFN score - just run health."""
    pos = [c for cls, nm, _, c in counts if cls.lower().startswith("pos")]
    neg = [c for cls, nm, _, c in counts if cls.lower().startswith("neg")]
    hk  = {nm: c for cls, nm, _, c in counts
           if cls.lower().startswith("house") or nm.upper() in HOUSEKEEPING}
    endo = [c for cls, nm, _, c in counts if cls.lower().startswith("endo")]

    neg_mean = float(np.mean(neg)) if neg else np.nan
    neg_sd   = float(np.std(neg, ddof=1)) if len(neg) > 1 else np.nan
    lod      = neg_mean + 2 * neg_sd if neg else np.nan

    return {
        "pos_control_geomean": _geomean(pos),
        "neg_control_mean":    neg_mean,
        "limit_of_detection":  lod,
        "hk_geomean":          _geomean(list(hk.values())),
        "n_housekeeping":      len(hk),
        "n_endogenous":        len(endo),
        "total_counts":        int(sum(endo)) if endo else 0,
        "n_genes_above_lod":   int(sum(1 for c in endo if not np.isnan(lod) and c > lod)) if endo else 0,
    }


# === CELL 4 : field parsing, specimen, record type ==========================

def parse_id_field(s):
    """Pull initials / patient id / date / plate position out of the ID string."""
    out = dict(sample_initials="", patient_id_raw="", sample_date_raw="", position="")
    if not s:
        return out
    s = s.strip()

    m = re.match(r"^(\d{2})\s+([ROCP]-?\d{3,5}|\d{4})$", s)      # "09 1013"
    if m:
        out["position"], out["patient_id_raw"] = m.group(1), m.group(2)
        return out
    if re.fullmatch(r"\d{1,2}", s):                               # bare position
        out["position"] = s
        return out

    mpos = re.match(r"^(\d{2})(?=[A-Za-z])", s)                   # "03JNZA-..."
    if mpos:
        out["position"] = mpos.group(1)

    m = INITIALS_LEAD.match(s)
    if m:
        out["sample_initials"] = m.group(1).upper()

    dates = [d for d in MMDDYY.findall(s)
             if 1 <= int(d[:2]) <= 12 and 1 <= int(d[2:4]) <= 31]
    if dates:
        out["sample_date_raw"] = dates[-1]

    for m in ID_IN_FIELD.finditer(s):
        tok = m.group(1)
        if tok not in dates and tok.upper() != out["sample_initials"]:
            out["patient_id_raw"] = tok.upper()
            break
    return out


def detect_specimen(blob):
    for name, pat in SPECIMEN_PATTERNS:
        if re.search(pat, blob, re.I):
            return name
    return DEFAULT_SPECIMEN


def detect_evidence(blob):
    return {
        "is_standard":     bool(STANDARD_PAT.search(blob)),
        "is_placeholder":  bool(PLACEHOLDER_PAT.search(blob)),
        "is_control":      bool(CONTROL_PAT.search(blob)),
        "is_other_cohort": bool(OTHER_COHORT_PAT.search(blob)),
        "is_covid":        bool(COVID_PAT.search(blob)),
        "is_stimulation":  bool(STIM_PAT.search(blob)),
        "has_timepoint":   bool(TIMEPOINT_PAT.search(blob)),
    }


# Any date outside this window is treated as a typo and dropped (-> NaT), so a
# bad year like 31 (-> 1931) or 30 (-> 2030) can never reach age or batch logic.
# Edit DATE_MIN if your earliest real run is earlier than 2008.
DATE_MIN = pd.Timestamp("2008-01-01")
DATE_MAX = pd.Timestamp.today().normalize()


def _in_window(d):
    return d if (pd.notna(d) and DATE_MIN <= d <= DATE_MAX) else pd.NaT


def parse_sample_date(mmddyy):
    """MMDDYY -> Timestamp. Two-digit years <=30 are 2000s.
    Returns NaT if the result is outside DATE_MIN..DATE_MAX."""
    if not mmddyy or len(mmddyy) != 6:
        return pd.NaT
    mm, dd, yy = int(mmddyy[:2]), int(mmddyy[2:4]), int(mmddyy[4:])
    year = 2000 + yy if yy <= 30 else 1900 + yy
    try:
        return _in_window(pd.Timestamp(year=year, month=mm, day=dd))
    except ValueError:
        return pd.NaT


def parse_run_date(raw, filename):
    for cand in (raw, (re.match(r"^(\d{8})", filename) or [None, ""])[0]):
        if not cand:
            continue
        for fmt in ("%Y%m%d", "%m/%d/%Y", "%Y-%m-%d", "%m/%d/%y"):
            try:
                d = _in_window(pd.Timestamp(datetime.strptime(str(cand).strip(), fmt)))
                if pd.notna(d):
                    return d
            except (ValueError, TypeError):
                continue
    return pd.NaT


FNAME_DATE8 = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")
FNAME_DATE6 = re.compile(r"(?<!\d)(\d{6})(?!\d)")


def _ymd(y, m, d):
    try:
        return _in_window(pd.Timestamp(year=y, month=m, day=d))
    except ValueError:
        return pd.NaT


def date_from_filename(filename):
    """Pull a date out of the RCC filename. YYYYMMDD first, then MMDDYY/YYMMDD."""
    stem = os.path.splitext(filename)[0]
    m = FNAME_DATE8.search(stem)
    if m:
        d = _ymd(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        if pd.notna(d):
            return d
    for m in FNAME_DATE6.finditer(stem):
        tok = m.group(1)
        d = parse_sample_date(tok)                               # MMDDYY
        if pd.isna(d):                                           # then YYMMDD
            yy, mm, dd = int(tok[:2]), int(tok[2:4]), int(tok[4:])
            d = _ymd(2000 + yy if yy <= 30 else 1900 + yy, mm, dd)
        if pd.notna(d):
            return d
    return pd.NaT


def resolve_sample_date(sample_date_raw):
    """Date the sample was COLLECTED. Only the RCC ID field carries this.

    The date in the filename is the NanoString run date, not the collection
    date - samples often sit frozen for months or years before being run - so
    it is deliberately not used here. If the ID field has no date, collection
    date is unknown and stays blank rather than being quietly faked.
    """
    d = parse_sample_date(sample_date_raw)
    return (d, "id_field") if pd.notna(d) else (pd.NaT, "none")


def resolve_run_date(run_date_raw, filename):
    """Date the cartridge was RUN. Instrument header first, then filename."""
    d = parse_run_date(run_date_raw, "")
    if pd.notna(d):
        return d, "rcc_date_header"
    d = date_from_filename(filename)
    if pd.notna(d):
        return d, "filename"
    return pd.NaT, "none"


def build_row(filename, folder):
    attrs, counts = parse_rcc(os.path.join(folder, filename))
    row = dict(raw_filename=filename, **attrs)
    row.update(parse_id_field(attrs["id_field_raw"]))

    blob = f"{attrs['id_field_raw']} {attrs['sample_comment']} {filename}"
    row["specimen_type"] = detect_specimen(blob)
    row.update(detect_evidence(blob))

    row["sample_date"], row["sample_date_source"] = resolve_sample_date(
        row["sample_date_raw"])
    row["run_date"], row["run_date_source"] = resolve_run_date(
        attrs["run_date_raw"], filename)

    ops = OP_FROM_CART.findall(
        re.sub(r"\d{8}|\d{2}-\d{2}-\d{2}", "", attrs["cartridge"]))
    row["operator"] = ops[-1] if ops else ""

    # numeric QC straight off the lane attributes
    try:
        row["binding_density"] = float(attrs["binding_density"])
    except (TypeError, ValueError):
        row["binding_density"] = np.nan
    try:
        row["fov_ratio"] = float(attrs["fov_counted"]) / float(attrs["fov_count"])
    except (TypeError, ValueError, ZeroDivisionError):
        row["fov_ratio"] = np.nan

    row.update(qc_from_counts(counts))
    return row, counts


# === CELL 5 : read every file ===============================================
files = sorted(f for f in os.listdir(FOLDER) if f.lower().endswith(".rcc"))
print(f"{len(files)} RCC files in {FOLDER}")

rows, long_counts, parse_errors = [], [], []
for i, fn in enumerate(files):
    try:
        row, counts = build_row(fn, FOLDER)
        rows.append(row)
        for cls, gene, acc, ct in counts:
            long_counts.append((fn, cls, gene, acc, ct))
    except Exception as exc:                                  # noqa: BLE001
        parse_errors.append({"raw_filename": fn, "error": f"{type(exc).__name__}: {exc}"})
    if i % 1000 == 0:
        print(f"  ...{i}/{len(files)}")

df = pd.DataFrame(rows)
errors_df = pd.DataFrame(parse_errors)

counts_long = pd.DataFrame(long_counts,
                           columns=["raw_filename", "code_class", "gene", "accession", "count"])
for col in ("code_class", "gene", "accession"):
    counts_long[col] = counts_long[col].astype("category")

print(f"\nparsed {len(df)} files, {len(errors_df)} failures, "
      f"{len(counts_long):,} probe measurements")
if not df.empty:
    print("\ncodesets present (GeneRLF):")
    print(df["gene_rlf"].value_counts().head(20))


# === CELL 6 : metadata ======================================================

def _find_col(columns, *wanted):
    """Match a column name ignoring case, spaces, and punctuation."""
    norm = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in columns}
    for w in wanted:
        key = re.sub(r"[^a-z0-9]", "", w.lower())
        if key in norm:
            return norm[key]
    return None


SHEET_EXTS = (".csv", ".xlsx", ".xls", ".tsv")


def _spreadsheets_in(d, max_depth=0):
    """Spreadsheet files in d, optionally descending max_depth levels."""
    found = []
    if not os.path.isdir(d):
        return found
    base_depth = d.rstrip(os.sep).count(os.sep)
    for root, dirs, files in os.walk(d):
        dirs[:] = [x for x in dirs if not x.startswith(".")]
        if root.rstrip(os.sep).count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
        for f in files:
            if f.lower().endswith(SHEET_EXTS) and not f.startswith("~$"):
                found.append(os.path.join(root, f))
    return found


def load_metadata(folder, pattern):
    """Find the metadata file without caring about capitalisation.

    Linux globbing is case-sensitive, so 'MetadataPatients_NoPHI*' misses
    'metadatapatients_nophi.xlsx'. This compares names with punctuation and
    case stripped out, looks in the RCC folder, its parent, and anything in
    METADATA_SEARCH_DIRS, and if it still finds nothing it tells you what
    spreadsheets it did see instead of leaving you guessing.
    """
    want = re.sub(r"[^a-z0-9]", "", pattern.lower().replace("*", ""))

    pool, seen = [], set()
    for d, depth in ([(folder, 0), (os.path.dirname(folder.rstrip(os.sep)), 1)]
                     + [(x, 2) for x in METADATA_SEARCH_DIRS]):
        for p in _spreadsheets_in(d, max_depth=depth):
            rp = os.path.realpath(p)
            if rp not in seen:
                seen.add(rp)
                pool.append(p)

    def key(p):
        return re.sub(r"[^a-z0-9]", "",
                      os.path.splitext(os.path.basename(p))[0].lower())

    hits = [p for p in pool if key(p).startswith(want)]
    if not hits:
        hits = [p for p in pool if want in key(p)]
    if not hits:                      # last resort: looks metadata-ish
        hits = [p for p in pool
                if ("metadata" in key(p)
                    or ("patient" in key(p) and "nanostring" not in key(p)))]

    if not hits:
        listing = "\n".join(f"    {p}" for p in sorted(pool)[:40]) or "    (none)"
        raise FileNotFoundError(
            f"No metadata file matching {pattern!r} near {folder}.\n"
            f"Spreadsheets I can see:\n{listing}\n"
            "Set METADATA_GLOB to the right name, or add its folder to "
            "METADATA_SEARCH_DIRS, in Cell 1.")

    # prefer a file sitting in FOLDER itself, then the shallowest path
    hits.sort(key=lambda p: (os.path.dirname(os.path.realpath(p))
                             != os.path.realpath(folder),
                             p.count(os.sep), p))
    path = hits[0]
    if len(hits) > 1:
        print(f"note: {len(hits)} candidates, using {path}")
        for p in hits[1:5]:
            print(f"      (ignoring {p})")

    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        sheets = pd.read_excel(path, sheet_name=None, dtype=str)
        # pick the sheet that actually looks like the patient table
        name, md = max(sheets.items(),
                       key=lambda kv: (sum(1 for c in kv[1].columns
                                           if re.search(r"enroll|diagnos|initial",
                                                        str(c), re.I)),
                                       len(kv[1])))
        if len(sheets) > 1:
            print(f"workbook has {len(sheets)} sheets, using {name!r}")
    elif ext == ".tsv":
        md = pd.read_csv(path, dtype=str, sep="\t")
    else:
        md = pd.read_csv(path, dtype=str)

    md.columns = [str(c).strip() for c in md.columns]
    md = md.dropna(how="all").reset_index(drop=True)
    print(f"metadata: {path}  ({len(md)} rows)")
    print("columns:", list(md.columns))
    return md, path


md, md_path = load_metadata(FOLDER, METADATA_GLOB)

if METADATA_EXTRA:
    _SYN = [("Enrollment ID", "EnrollmentID"),
            ("Alternate Enrollment ID", "Alt Enrollment ID"),
            ("Initials",), ("Birth Date", "DOB", "BirthDate"),
            ("Sex", "Gender"), ("Race",), ("Ethnicity",),
            ("Study Status", "Status"), ("Diagnosis1", "Diagnosis 1"),
            ("Diagnosis2", "Diagnosis 2"), ("Note", "Notes")]
    _parts = [md]
    for _p in METADATA_EXTRA:
        if not os.path.exists(_p):
            raise FileNotFoundError(f"METADATA_EXTRA file not found: {_p}")
        if _p.lower().endswith((".xlsx", ".xls")):
            _ex = pd.read_excel(_p, dtype=str)
        elif _p.lower().endswith(".tsv"):
            _ex = pd.read_csv(_p, dtype=str, sep="\t")
        else:
            _ex = pd.read_csv(_p, dtype=str)
        _ex.columns = [str(c).strip() for c in _ex.columns]
        _ex = _ex.dropna(how="all")
        _ren = {}
        for _names in _SYN:
            _src, _dst = _find_col(_ex.columns, *_names), _find_col(md.columns, *_names)
            if _src and _dst and _src != _dst:
                _ren[_src] = _dst
        _ex = _ex.rename(columns=_ren).reindex(columns=md.columns)
        print(f"extra patients: {_p}  (+{len(_ex)} rows)")
        _parts.append(_ex)
    md = pd.concat(_parts, ignore_index=True)
    print(f"metadata now {len(md)} rows (main sheet first, so it wins on duplicate IDs)")

COL = {
    "enroll":     _find_col(md.columns, "Enrollment ID", "EnrollmentID"),
    "alt_enroll": _find_col(md.columns, "Alternate Enrollment ID", "Alt Enrollment ID"),
    "initials":   _find_col(md.columns, "Initials"),
    "dob":        _find_col(md.columns, "Birth Date", "DOB", "BirthDate"),
    "sex":        _find_col(md.columns, "Sex", "Gender"),
    "race":       _find_col(md.columns, "Race"),
    "ethnicity":  _find_col(md.columns, "Ethnicity"),
    "status":     _find_col(md.columns, "Study Status", "Status"),
    "dx1":        _find_col(md.columns, "Diagnosis1", "Diagnosis 1"),
    "dx2":        _find_col(md.columns, "Diagnosis2", "Diagnosis 2"),
    "note":       _find_col(md.columns, "Note", "Notes"),
}
missing = [k for k, v in COL.items() if v is None]
if missing:
    print(f"warning: metadata columns not found, will be blank: {missing}")

# --- lookup on both enrollment ID columns -----------------------------------
meta_lookup, dup_keys = {}, []
for idx, r in md.iterrows():
    for key_col in (COL["enroll"], COL["alt_enroll"]):
        if not key_col:
            continue
        k = normalize_id(r.get(key_col))
        if not k:
            continue
        if k in meta_lookup and meta_lookup[k] != idx:
            dup_keys.append(k)
        meta_lookup.setdefault(k, idx)
if dup_keys:
    print(f"warning: {len(set(dup_keys))} enrollment IDs appear on >1 metadata row "
          f"(first wins): {sorted(set(dup_keys))[:10]}")

# --- initials -> enrollment ID, derived from the metadata file --------------
_by_initials = {}
if COL["initials"] and COL["enroll"]:
    for _, r in md.iterrows():
        ini = str(r.get(COL["initials"]) or "").strip().upper()
        eid = normalize_id(r.get(COL["enroll"]))
        if ini and eid:
            _by_initials.setdefault(ini, set()).add(eid)

AUTO_INITIALS_MAP = {k: next(iter(v)) for k, v in _by_initials.items() if len(v) == 1}
AMBIGUOUS_INITIALS = {k for k, v in _by_initials.items() if len(v) > 1} | KNOWN_AMBIGUOUS

# manual entries win
INITIALS_MAP = {**AUTO_INITIALS_MAP, **MANUAL_INITIALS_MAP}
print(f"\ninitials map: {len(AUTO_INITIALS_MAP)} auto from metadata, "
      f"{len(MANUAL_INITIALS_MAP)} manual overrides, "
      f"{len(AMBIGUOUS_INITIALS)} ambiguous (never auto-filled)")


# === CELL 7 : resolve patient IDs ===========================================
# A 1-to-1 initials->ID map learned from files that state their own ID.
_seen = {}
for _, r in df.iterrows():
    pid = normalize_id(r["patient_id_raw"])
    ini = r["sample_initials"]
    if ini and pid and not r["is_standard"] and not r["is_placeholder"]:
        _seen.setdefault(ini, set()).add(pid)
FILE_LEARNED_MAP = {k: next(iter(v)) for k, v in _seen.items()
                    if len(v) == 1 and k not in AMBIGUOUS_INITIALS}


def resolve_identity(r):
    """-> (patient_id, source, needs_review, review_reason)"""
    ini = r["sample_initials"]
    pid = normalize_id(r["patient_id_raw"])

    if (ini, pid) in VERIFIED_TYPOS:
        return VERIFIED_TYPOS[(ini, pid)], "verified_typo_correction", False, ""

    if pid:
        expected = INITIALS_MAP.get(ini)
        if expected and expected != pid:
            return (pid, "stated_in_file", True,
                    f"file says {pid}, metadata says {ini} is {expected}")
        return pid, "stated_in_file", False, ""

    # no ID written in the file: fill in only where it is unambiguous
    if not ini:
        return "", "none", False, ""
    if ini in AMBIGUOUS_INITIALS:
        return "", "none", True, f"initials {ini} map to more than one patient, no ID in file"
    for src, mapping in (("manual_map", MANUAL_INITIALS_MAP),
                         ("metadata_initials", AUTO_INITIALS_MAP),
                         ("learned_from_files", FILE_LEARNED_MAP)):
        if ini in mapping:
            return mapping[ini], src, False, ""
    return "", "none", False, ""


res = df.apply(lambda r: pd.Series(resolve_identity(r),
                                   index=["patient_id", "id_source",
                                          "needs_review", "review_reason"]), axis=1)
df = pd.concat([df, res], axis=1)

# --- join metadata ----------------------------------------------------------
df["meta_row"] = df["patient_id"].map(lambda p: meta_lookup.get(p, -1) if p else -1)
df["metadata_matched"] = df["meta_row"] >= 0


def meta_value(meta_row, key):
    col = COL.get(key)
    if meta_row < 0 or not col:
        return ""
    val = md.at[meta_row, col]
    return "" if pd.isna(val) else str(val).strip()


for field in ("initials", "dob", "sex", "race", "ethnicity", "status", "dx1", "dx2", "note"):
    df[f"meta_{field}"] = df["meta_row"].map(lambda i, f=field: meta_value(i, f))

df = df.rename(columns={
    "meta_sex": "sex", "meta_race": "race", "meta_ethnicity": "ethnicity",
    "meta_status": "study_status", "meta_dx1": "diagnosis1",
    "meta_dx2": "diagnosis2", "meta_note": "metadata_note",
})

# --- age at sample date -----------------------------------------------------
# Birth date is used here and then discarded, so it never reaches an output.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    dob = pd.to_datetime(df["meta_dob"], errors="coerce")

df["age_at_sample_yrs"] = ((df["sample_date"] - dob).dt.days / 365.25).round(1)
df.loc[(df["age_at_sample_yrs"] < 0) | (df["age_at_sample_yrs"] > 120),
       "age_at_sample_yrs"] = np.nan

# why age is blank, so a gap is explainable rather than mysterious
df["age_unavailable_reason"] = ""
df.loc[df["age_at_sample_yrs"].isna() & dob.isna() & df["metadata_matched"],
       "age_unavailable_reason"] = "no birth date in metadata"
df.loc[df["age_at_sample_yrs"].isna() & df["sample_date"].isna(),
       "age_unavailable_reason"] = "no collection date (only a run date)"
df.loc[df["age_at_sample_yrs"].isna() & ~df["metadata_matched"],
       "age_unavailable_reason"] = "patient not matched to metadata"

n_no_dob = int((df["age_unavailable_reason"] == "no birth date in metadata").sum())
n_no_date = int((df["age_unavailable_reason"] == "no collection date (only a run date)").sum())
print(f"\nage computed for {df['age_at_sample_yrs'].notna().sum()}/{len(df)} files"
      f"  (no DOB: {n_no_dob}, no collection date: {n_no_date})")
print("collection-date source:")
print(df["sample_date_source"].value_counts().to_string())
print("run-date source:")
print(df["run_date_source"].value_counts().to_string())

if DROP_BIRTH_DATE:
    df = df.drop(columns=["meta_dob"])
    print("birth date dropped from the table - only age_at_sample_yrs remains")

# initials disagreement is worth flagging even when the ID matched
mismatch = (df["metadata_matched"] & (df["sample_initials"] != "")
            & (df["meta_initials"].str.upper() != "")
            & (df["sample_initials"] != df["meta_initials"].str.upper()))
df.loc[mismatch, "needs_review"] = True
df.loc[mismatch, "review_reason"] = df.loc[mismatch].apply(
    lambda r: (r["review_reason"] + "; " if r["review_reason"] else "")
              + f"initials {r['sample_initials']} != metadata {r['meta_initials']}", axis=1)

print(f"\npatient ID resolved: {(df['patient_id'] != '').sum()}/{len(df)}")
print(f"matched to metadata: {df['metadata_matched'].sum()}")
print(f"flagged for review:  {df['needs_review'].sum()}")


# === CELL 8 : record type and QC verdict ====================================

def record_type(r):
    if r["is_placeholder"]:
        return "placeholder_or_test"
    if r["is_standard"]:
        return "standard"
    if r["metadata_matched"]:
        return "patient_sample"
    if r["patient_id"]:
        return "patient_id_not_in_metadata"
    if r["is_covid"]:
        return "covid_project"
    if r["is_other_cohort"]:
        return "other_cohort"
    if r["is_control"]:
        return "healthy_control"
    if r["is_stimulation"]:
        return "in_vitro_experiment"
    if r["specimen_type"] != DEFAULT_SPECIMEN:
        return "non_blood_unidentified"
    return "unclassified"


df["record_type"] = df.apply(record_type, axis=1)
df["is_non_blood"] = df["specimen_type"] != DEFAULT_SPECIMEN


def qc_verdict(r):
    problems = []
    if pd.notna(r["binding_density"]) and not (BD_MIN <= r["binding_density"] <= BD_MAX):
        problems.append(f"binding_density={r['binding_density']:.2f}")
    if pd.notna(r["fov_ratio"]) and r["fov_ratio"] < FOV_MIN:
        problems.append(f"fov_ratio={r['fov_ratio']:.2f}")
    if r["n_probes"] == 0:
        problems.append("no_counts_in_file")
    if pd.notna(r["hk_geomean"]) and r["hk_geomean"] < 50:
        problems.append(f"low_housekeeping={r['hk_geomean']:.0f}")
    return "; ".join(problems)


df["qc_problems"] = df.apply(qc_verdict, axis=1)
df["qc_pass"] = df["qc_problems"] == ""

print(df["record_type"].value_counts())
print()
print("specimen types:")
print(df["specimen_type"].value_counts())
print(f"\nQC pass: {df['qc_pass'].sum()}/{len(df)}")


# === CELL 9 : readable filenames ============================================

# visit number per patient, ordered by sample date
# Collection date where we have it, run date only as a last resort for
# ordering. filename_date_is records which one ended up in the new name.
df["_sort_date"] = df["sample_date"].fillna(df["run_date"])
df["filename_date_is"] = np.where(df["sample_date"].notna(), "collection",
                          np.where(df["run_date"].notna(), "run_date", "none"))
df["visit_num"] = (
    df[df["patient_id"] != ""]
    .sort_values(["patient_id", "_sort_date", "raw_filename"])
    .groupby("patient_id")
    .cumcount() + 1
).reindex(df.index)


def build_new_name(r):
    pid = r["patient_id"] or "NOID"
    parts = [pid]
    if INCLUDE_INITIALS and r["sample_initials"]:
        parts.append(r["sample_initials"])

    d = r["_sort_date"]
    parts.append(d.strftime("%Y%m%d") if pd.notna(d) else "NODATE")

    if INCLUDE_DX:
        parts.append(slug(r["diagnosis1"]) or "NODX")
    if INCLUDE_SPECIMEN and r["specimen_type"] != DEFAULT_SPECIMEN:
        parts.append(slug(r["specimen_type"]))
    if INCLUDE_VISIT and pd.notna(r["visit_num"]):
        parts.append(f"v{int(r['visit_num']):02d}")
    if r["record_type"] != "patient_sample":
        parts.append(slug(r["record_type"]))
    if INCLUDE_LANE and r["lane_id"]:
        parts.append(f"L{str(r['lane_id']).zfill(2)}")
    return "_".join(p for p in parts if p)


df["new_filename"] = df.apply(build_new_name, axis=1) + ".RCC"

# deterministic collision suffix
dupe = df["new_filename"].duplicated(keep=False)
if dupe.any():
    order = df[dupe].sort_values("raw_filename").groupby("new_filename").cumcount() + 1
    df.loc[order.index, "new_filename"] = (
        df.loc[order.index, "new_filename"].str.replace(r"\.RCC$", "", regex=True)
        + "_d" + order.astype(str) + ".RCC")
    print(f"{dupe.sum()} filename collisions resolved with _dN suffixes")

rename_map = df[["raw_filename", "new_filename", "patient_id", "diagnosis1",
                 "specimen_type", "record_type", "needs_review"]].copy()
print(f"\nexample new names:")
for _, r in rename_map.head(8).iterrows():
    print(f"  {r['raw_filename'][:45]:45s} -> {r['new_filename']}")


# === CELL 10 : optionally copy the files ====================================
if RENAME_MODE in ("copy_flat", "copy_tree"):
    dest_root = os.path.join(OUT, "renamed")
    os.makedirs(dest_root, exist_ok=True)
    copied = 0
    for _, r in df.iterrows():
        if RENAME_MODE == "copy_tree":
            sub = os.path.join(dest_root,
                               slug(r["diagnosis1"]) or "NODX",
                               r["patient_id"] or "NOID")
        else:
            sub = dest_root
        os.makedirs(sub, exist_ok=True)
        shutil.copy2(os.path.join(FOLDER, r["raw_filename"]),
                     os.path.join(sub, r["new_filename"]))
        copied += 1
    print(f"copied {copied} files into {dest_root}")
else:
    print("RENAME_MODE='map_only' - no files were copied. "
          "Review nanostring_rename_map.csv, then set RENAME_MODE and re-run this cell.")


# === CELL 11 : wide count matrices ==========================================
counts_long = counts_long.merge(
    df[["raw_filename", "patient_id", "new_filename", "record_type", "specimen_type"]],
    on="raw_filename", how="left")

panel_genes = [g for g in TARGET_GENES + HOUSEKEEPING
               if g in set(counts_long["gene"].astype(str))]
absent = [g for g in TARGET_GENES + HOUSEKEEPING if g not in panel_genes]
if absent:
    print(f"warning: {len(absent)} target genes never appear in any file: {absent}")

wide_target = (counts_long[counts_long["gene"].astype(str).isin(panel_genes)]
               .pivot_table(index="raw_filename", columns="gene", values="count",
                            aggfunc="first", observed=True)
               .reindex(columns=panel_genes))
wide_target.columns = [str(c) for c in wide_target.columns]
wide_target = wide_target.reset_index()
print(f"\ntarget-gene matrix: {wide_target.shape[0]} samples x {len(panel_genes)} genes")

# full wide matrix, all probes, only if it will not explode
n_probes_total = counts_long["gene"].nunique()
wide_all = None
if n_probes_total <= 5000:
    wide_all = (counts_long.pivot_table(index="raw_filename", columns="gene",
                                        values="count", aggfunc="first", observed=True)
                .reset_index())
    wide_all.columns = [str(c) for c in wide_all.columns]
    print(f"full matrix: {wide_all.shape[0]} x {n_probes_total} probes")
else:
    print(f"{n_probes_total} distinct probes - skipping the full wide matrix, "
          "use the long file instead")


# === CELL 12 : assemble the single master table =============================
# One row per RCC file. Everything that used to be a separate sheet is now a
# column you can filter on: record_type, specimen_type, needs_review, qc_pass.
IDENTITY_COLS = [
    # who
    "new_filename", "patient_id", "meta_initials", "sample_initials",
    # when / how old. sample_date is COLLECTION date; run_date is when the
    # cartridge was run. They can be years apart.
    "sample_date", "sample_date_source", "age_at_sample_yrs",
    "age_unavailable_reason", "visit_num", "filename_date_is",
    # demographics, straight from MetadataPatients_NoPHI (no birth date)
    "sex", "race", "ethnicity", "diagnosis1", "diagnosis2", "study_status",
    # what kind of sample
    "specimen_type", "is_non_blood", "record_type",
    "has_timepoint", "is_stimulation",
    # how confident we are about the identity
    "needs_review", "review_reason", "id_source", "metadata_matched",
    # run / batch
    "run_date", "run_date_source", "cartridge", "lane_id", "operator", "position",
    "gene_rlf", "n_probes",
    # QC
    "binding_density", "fov_ratio", "pos_control_geomean", "neg_control_mean",
    "limit_of_detection", "hk_geomean", "n_housekeeping", "n_genes_above_lod",
    "total_counts", "qc_pass", "qc_problems",
    # provenance
    "id_field_raw", "sample_comment", "metadata_note", "raw_filename",
]
if DROP_INITIALS:
    IDENTITY_COLS = [c for c in IDENTITY_COLS
                     if c not in ("meta_initials", "sample_initials")]
IDENTITY_COLS = [c for c in IDENTITY_COLS if c in df.columns]

_counts_block = wide_target
if INCLUDE_ALL_PROBES and wide_all is not None:
    _counts_block = wide_all
    _clash = [c for c in _counts_block.columns
              if c != "raw_filename" and c in IDENTITY_COLS]
    if _clash:                                   # never overwrite an annotation
        _counts_block = _counts_block.rename(columns={c: f"{c}_count" for c in _clash})
    # panel genes first (same order as before), then every other probe
    _first = [g for g in panel_genes if g in _counts_block.columns]
    _rest = [c for c in _counts_block.columns
             if c not in _first and c != "raw_filename"]
    _counts_block = _counts_block[["raw_filename"] + _first + _rest]
COUNT_COLS = [c for c in _counts_block.columns if c != "raw_filename"]
final = df[IDENTITY_COLS].merge(_counts_block, on="raw_filename", how="left")

# put raw_filename last
final = final[[c for c in final.columns if c != "raw_filename"] + ["raw_filename"]]
for c in ("sample_date", "run_date"):
    if c in final.columns:
        final[c] = final[c].dt.strftime("%Y-%m-%d").fillna("")

# hard guarantee: no birth date in the output, whatever happened upstream
if DROP_BIRTH_DATE:
    leaked = [c for c in final.columns
              if re.search(r"birth|\bdob\b", str(c), re.I)]
    if leaked:
        final = final.drop(columns=leaked)
        print(f"removed birth-date columns from output: {leaked}")

assert final["raw_filename"].is_unique, "duplicate rows per file - investigate"
print(f"master table: {final.shape[0]} rows x {final.shape[1]} columns")


# === CELL 13 : write everything (CSV only) ==================================
# One master CSV, every file and every column. The old sheet splits are
# columns now, so filter in pandas (or Excel after opening):
#   record_type == "patient_sample"  -> the analysis set
#   record_type == "standard"        -> the standards
#   needs_review == True             -> the identity conflicts to resolve
#   is_non_blood == True             -> BAL, PBMC, sorted cells, tissue
summary = pd.DataFrame({
    "metric": [
        "RCC files found", "parsed successfully", "parse failures",
        "patient ID resolved", "matched to metadata", "flagged needs_review",
        "non-blood specimens", "standards", "QC pass",
        "age computed", "no birth date in metadata", "no usable date on file",
        "unique patients", "unique diagnoses", "distinct probes",
        "date range",
    ],
    "value": [
        len(files), len(df), len(errors_df),
        int((df["patient_id"] != "").sum()), int(df["metadata_matched"].sum()),
        int(df["needs_review"].sum()), int(df["is_non_blood"].sum()),
        int((df["record_type"] == "standard").sum()), int(df["qc_pass"].sum()),
        int(df["age_at_sample_yrs"].notna().sum()), n_no_dob, n_no_date,
        df.loc[df["patient_id"] != "", "patient_id"].nunique(),
        df.loc[df["diagnosis1"] != "", "diagnosis1"].nunique(),
        n_probes_total,
        f"{df['_sort_date'].min():%Y-%m-%d} to {df['_sort_date'].max():%Y-%m-%d}",
    ],
})

# --- a column dictionary ----------------------------------------------------
COMPARISON_NAMES = {
    "patient_id": "Pt ID / NIH Enroll ID", "meta_initials": "Pt Initials",
    "sample_initials": "nanostring initials", "sample_date": "nanostring date",
    "age_at_sample_yrs": "age (yrs)", "sex": "gender", "race": "race1",
    "cartridge": "cartridge sample #", "raw_filename": "Process ID",
}
dictionary = pd.DataFrame({
    "column": list(final.columns),
    "kind": ["raw count" if c in set(COUNT_COLS) else "annotation"
             for c in final.columns],
    "matches_comparison_column": [COMPARISON_NAMES.get(c, "") for c in final.columns],
    "non_empty_rows": [int(final[c].notna().sum()) for c in final.columns],
})

# --- flat files -------------------------------------------------------------
final.to_csv(os.path.join(OUT, "nanostring_master.csv"), index=False)
summary.to_csv(os.path.join(OUT, "nanostring_summary.csv"), index=False)
dictionary.to_csv(os.path.join(OUT, "nanostring_column_dictionary.csv"), index=False)
rename_map.to_csv(os.path.join(OUT, "nanostring_rename_map.csv"), index=False)
if not errors_df.empty:
    errors_df.to_csv(os.path.join(OUT, "nanostring_parse_errors.csv"), index=False)
if wide_all is not None:
    wide_all.to_csv(os.path.join(OUT, "nanostring_counts_wide_all_probes.csv"), index=False)

long_path = os.path.join(OUT, "nanostring_counts_long.parquet")
try:
    counts_long.to_parquet(long_path, index=False)
except Exception:                                             # noqa: BLE001
    long_path = os.path.join(OUT, "nanostring_counts_long.csv.gz")
    counts_long.to_csv(long_path, index=False, compression="gzip")

print("\n" + summary.to_string(index=False))
print(f"\nwritten to {OUT}:")
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f"  {f:46s} {os.path.getsize(p)/1e6:8.1f} MB")

print("\nfilter nanostring_master.csv on these to get the old sub-tables:")
for col in ("record_type", "specimen_type"):
    print(f"\n  {col}")
    for k, v in final[col].value_counts().items():
        print(f"    {k:30s} {v:>7,}")
print(f"\n  needs_review = True            {int(final['needs_review'].sum()):>7,}")
print(f"  is_non_blood = True            {int(final['is_non_blood'].sum()):>7,}")
print(f"  qc_pass = False                {int((~final['qc_pass']).sum()):>7,}")


# === CELL 13b : one row per PATIENT - metadata + age, no birth dates ========
# nanostring_master.csv is one row per RCC *file*. This is the other view:
# one row per person in the metadata sheet (including people with no RCC file
# yet), with diagnosis, demographics, age and a summary of their samples.
# Birth date is used only to compute age and is not written anywhere.
_dob_col = COL["dob"]
pt = md.copy()
_dob = (pd.to_datetime(pt[_dob_col], errors="coerce") if _dob_col
        else pd.Series(pd.NaT, index=pt.index))
_now = pd.Timestamp.today().normalize()
pt["age_now_yrs"] = ((_now - _dob).dt.days / 365.25).round(1)
pt.loc[(pt["age_now_yrs"] < 0) | (pt["age_now_yrs"] > 120), "age_now_yrs"] = np.nan

_g = df[df["meta_row"] >= 0].groupby("meta_row")
_stats = pd.DataFrame({
    "n_rcc_files":      _g.size(),
    "n_whole_blood":    _g["specimen_type"].apply(lambda s: int((s == "whole_blood").sum())),
    "first_sample_date": _g["sample_date"].min(),
    "last_sample_date":  _g["sample_date"].max(),
    "first_run_date":    _g["run_date"].min(),
    "last_run_date":     _g["run_date"].max(),
    "age_at_first_sample_yrs": _g["age_at_sample_yrs"].min(),
    "age_at_last_sample_yrs":  _g["age_at_sample_yrs"].max(),
})
pt = pt.join(_stats)
pt["n_rcc_files"] = pt["n_rcc_files"].fillna(0).astype(int)
pt["n_whole_blood"] = pt["n_whole_blood"].fillna(0).astype(int)

_key = pt[COL["enroll"]].map(normalize_id) if COL["enroll"] else pd.Series("", index=pt.index)
pt["duplicate_enrollment_id"] = (_key != "") & _key.duplicated(keep=False)

for c in ("first_sample_date", "last_sample_date", "first_run_date", "last_run_date"):
    pt[c] = pt[c].dt.strftime("%Y-%m-%d").fillna("")

# no birth dates in the output, whatever the source column is called
pt = pt.drop(columns=[c for c in pt.columns
                      if re.search(r"birth|\bdob\b", str(c), re.I)])
if DROP_INITIALS and COL["initials"] in pt.columns:
    pt = pt.drop(columns=[COL["initials"]])

if COL["enroll"]:
    pt = pt.sort_values(COL["enroll"], key=lambda s: s.astype(str).str.upper())

pt_csv = os.path.join(OUT, "nanostring_patients.csv")
pt.to_csv(pt_csv, index=False)

print(f"\npatients table: {len(pt)} people, "
      f"{int((pt['n_rcc_files'] > 0).sum())} with >=1 RCC file, "
      f"{int(pt['age_now_yrs'].notna().sum())} with an age")
print(f"  -> {pt_csv}")
print("  columns:", list(pt.columns))


# === CELL 14 : what are the standards actually good for? ====================
# A standard / calibration tube is the SAME pooled RNA run again and again on
# different cartridges. Biologically it never changes, so any difference in
# its counts between cartridges is pure batch effect - which is exactly what
# lets you correct for batch.
#
# Three different corrections get confused with each other. They stack:
#   1. POS_A..POS_F spike-ins  -> hybridisation efficiency, within one lane
#   2. housekeeping geomean    -> how much RNA went in, within one sample
#   3. the standard tube       -> drift between cartridges and over years
# Only #3 needs these files. #1 and #2 are already computed per sample above.
#
# Before calibrating anything you need to know two things about your
# standards, and neither can be assumed. This cell answers both.

std = df[df["record_type"] == "standard"].copy()
n_cart = df["cartridge"].replace("", np.nan).nunique()
cart_with_std = std["cartridge"].replace("", np.nan).nunique()

print("=" * 70)
print("STANDARDS DIAGNOSTIC")
print("=" * 70)
print(f"standard files            : {len(std)}")
print(f"distinct cartridges, all  : {n_cart}")
print(f"cartridges with a standard: {cart_with_std}"
      f"  ({cart_with_std / n_cart * 100:.1f}%)" if n_cart else "")

# --- Q1: can you calibrate per cartridge at all? ----------------------------
if n_cart and cart_with_std / n_cart < 0.5:
    print("\n>> Most cartridges have NO standard on them. Per-cartridge")
    print("   calibration is not possible for the bulk of the data. Options,")
    print("   in order of how defensible they are:")
    print("     a) calibrate only the subset of cartridges that do have one")
    print("     b) treat run_date as the batch variable and model it")
    print("     c) look for unlabelled standards (candidates listed below)")
else:
    print("\n>> Enough coverage to calibrate per cartridge.")

# --- Q2: is it the same reference material throughout? ----------------------
# Reference RNA lots get used up and replaced. A new lot is a step change in
# the standard's counts, and calibrating across one would inject a fake
# batch effect rather than remove it.
std_counts = counts_long[
    counts_long["raw_filename"].isin(std["raw_filename"])
    & counts_long["code_class"].astype(str).str.lower().str.startswith(("endo", "house"))
].copy()

std_profile = pd.DataFrame()
if not std_counts.empty:
    g = (std_counts.groupby("raw_filename", observed=True)["count"]
         .apply(lambda s: _geomean(s.tolist())).rename("geomean_all_genes"))
    std_profile = (std[["raw_filename", "cartridge", "run_date", "id_field_raw",
                        "hk_geomean", "pos_control_geomean", "qc_pass"]]
                   .merge(g, on="raw_filename", how="left")
                   .sort_values("run_date"))
    std_profile["year"] = std_profile["run_date"].dt.year

    print("\nstandard signal by year (geomean across all genes):")
    yr = (std_profile.groupby("year")["geomean_all_genes"]
          .agg(n="size", median="median",
               cv=lambda s: s.std() / s.mean() if len(s) > 1 and s.mean() else np.nan))
    print(yr.to_string(float_format=lambda v: f"{v:,.2f}"))

    overall_cv = (std_profile["geomean_all_genes"].std()
                  / std_profile["geomean_all_genes"].mean()
                  if std_profile["geomean_all_genes"].mean() else np.nan)
    print(f"\noverall CV across all standards: {overall_cv:.1%}")

    # A new reference lot shows up as a step change, not as general noise, so
    # look for the year boundary that best splits the series in two. Scored on
    # the log scale with a t-like statistic, so a big ratio resting on two
    # noisy points does not beat a smaller but cleanly separated step.
    MIN_PER_SIDE = 3
    steps = []
    lvl = np.log(std_profile["geomean_all_genes"].where(
        std_profile["geomean_all_genes"] > 0))
    yrs = sorted(std_profile["year"].dropna().unique())
    for cut in yrs[1:]:
        before = lvl[std_profile["year"] < cut].dropna()
        after = lvl[std_profile["year"] >= cut].dropna()
        if len(before) < MIN_PER_SIDE or len(after) < MIN_PER_SIDE:
            continue
        se = np.sqrt(before.var(ddof=1) / len(before) + after.var(ddof=1) / len(after))
        diff = after.mean() - before.mean()
        t = abs(diff) / se if se and np.isfinite(se) else np.nan
        steps.append((int(cut), float(np.exp(diff)), float(t), len(before), len(after)))

    if steps:
        steps.sort(key=lambda s: -(s[2] if np.isfinite(s[2]) else 0))
        cut, ratio, t, nb, na = steps[0]
        print(f"largest step change: {cut} ({nb} standards before, {na} after), "
              f"level {ratio:.2f}x, separation t={t:.1f}")
        if t >= 3 and (ratio > 1.3 or ratio < 0.77):
            print(">> That is a step, not drift. Most likely a new reference lot")
            print(f"   or a protocol change. Set ERA_BREAKS = ['{cut}-01-01'] in")
            print("   Cell 15 so factors are computed within each era separately.")
            print("   Calibrating across a step injects a fake batch effect.")
        else:
            print(">> No clean step. Consistent with one reference material.")
        print("\n   all candidate split years (year, ratio, t):")
        for c, r, tt, _, _ in sorted(steps):
            print(f"     {c}  {r:5.2f}x  t={tt:4.1f}")
    elif pd.notna(overall_cv) and overall_cv > 0.5:
        print(">> High spread but too few standards to localise a step change.")

# --- Q3: are there standards you never labelled as such? --------------------
# An ID string that shows up on many different cartridges is behaving like a
# standard whether or not anyone called it one.
rep = (df[df["id_field_raw"].str.strip() != ""]
       .groupby("id_field_raw")
       .agg(files=("raw_filename", "size"),
            cartridges=("cartridge", pd.Series.nunique),
            labelled_standard=("record_type", lambda s: (s == "standard").any()),
            span_days=("run_date", lambda s: (s.max() - s.min()).days
                       if s.notna().sum() > 1 else 0))
       .reset_index())
candidates = (rep[(rep["cartridges"] >= 5) & (~rep["labelled_standard"])]
              .sort_values("cartridges", ascending=False))
print(f"\nunlabelled IDs appearing on 5+ cartridges: {len(candidates)}")
if not candidates.empty:
    print(candidates.head(15).to_string(index=False))
    print(">> Check these. Any that are really the same pooled RNA should be")
    print("   added to STANDARD_PAT in Cell 2 and the pipeline re-run.")

std_profile.to_csv(os.path.join(OUT, "standards_profile.csv"), index=False)
candidates.to_csv(os.path.join(OUT, "standards_candidates_unlabelled.csv"), index=False)


# === CELL 15 : per-cartridge calibration factors ============================
# For each cartridge that carries a standard, and each gene:
#     factor = (reference level for that gene) / (this cartridge's standard)
# Multiply a patient sample's counts by its cartridge's factor and the
# cartridge-to-cartridge drift cancels out. The reference level is the median
# across all standards in the same era, so factors centre on 1.
#
# Factors are written out but NOT applied. Look at them first - a factor far
# from 1 means that cartridge was an outlier, and you want to know why before
# you silently rescale patient data with it.
APPLY_CALIBRATION = False
ERA_BREAKS = []          # e.g. ["2018-01-01"] to split at a known lot change

calib = pd.DataFrame()
if not std_counts.empty and cart_with_std:
    sc = std_counts.merge(df[["raw_filename", "cartridge", "run_date"]],
                          on="raw_filename", how="left")
    sc = sc[sc["cartridge"].astype(str).str.strip() != ""]
    sc["gene"] = sc["gene"].astype(str)      # drop categorical, it breaks reindex

    breaks = [pd.Timestamp.min] + [pd.Timestamp(b) for b in ERA_BREAKS] + [pd.Timestamp.max]
    sc["era"] = pd.cut(sc["run_date"], bins=breaks, labels=False, include_lowest=True)

    # one value per cartridge x gene (average if a cartridge ran 2 standards)
    per_cart = (sc.groupby(["era", "cartridge", "gene"], observed=True)["count"]
                .mean().reset_index())
    # reference = median across cartridges, within era and gene
    ref = (per_cart.groupby(["era", "gene"], observed=True)["count"]
           .median().rename("reference").reset_index())

    calib = per_cart.merge(ref, on=["era", "gene"], how="left")
    calib["factor"] = np.where(calib["count"] > 0,
                               calib["reference"] / calib["count"], np.nan)
    calib = calib.rename(columns={"count": "standard_count"})

    spread = (calib.groupby("cartridge")["factor"]
              .agg(median_factor="median", min_factor="min", max_factor="max")
              .sort_values("median_factor"))
    print("\n" + "=" * 70)
    print("CALIBRATION FACTORS")
    print("=" * 70)
    print(f"{calib['cartridge'].nunique()} cartridges x {calib['gene'].nunique()} genes")
    print("\ncartridges furthest from the reference:")
    print(pd.concat([spread.head(5), spread.tail(5)])
          .to_string(float_format=lambda v: f"{v:,.2f}"))

    outliers = spread[(spread["median_factor"] < 0.5) | (spread["median_factor"] > 2.0)]
    if len(outliers):
        print(f"\n>> {len(outliers)} cartridges are off by more than 2x. Investigate")
        print("   these before applying anything - a failed standard lane will")
        print("   otherwise drag every patient sample on that cartridge with it.")

    calib.to_csv(os.path.join(OUT, "calibration_factors.csv"), index=False)

    if APPLY_CALIBRATION:
        cf = calib.set_index(["cartridge", "gene"])["factor"]
        cal_long = counts_long.merge(df[["raw_filename", "cartridge"]],
                                     on="raw_filename", how="left")
        idx = pd.MultiIndex.from_arrays(
            [cal_long["cartridge"].astype(str), cal_long["gene"].astype(str)])
        cal_long["factor"] = cf.reindex(idx).values
        cal_long["count_calibrated"] = cal_long["count"] * cal_long["factor"].fillna(1.0)
        cal_long["was_calibrated"] = cal_long["factor"].notna()
        cal_long.to_csv(os.path.join(OUT, "counts_calibrated_long.csv.gz"),
                        index=False, compression="gzip")
        print(f"\napplied to {cal_long['was_calibrated'].sum():,} of "
              f"{len(cal_long):,} measurements "
              f"({cal_long['was_calibrated'].mean():.1%} had a factor available)")
    else:
        print("\nAPPLY_CALIBRATION is False - factors written, counts untouched.")
else:
    print("\nno usable standards with cartridge IDs, skipping calibration")

Note: you may need to restart the kernel to use updated packages.
7690 RCC files in /home/jovyan/rcc_extracted/Nanostring_RCC
  ...0/7690
  ...1000/7690
  ...2000/7690
  ...3000/7690
  ...4000/7690
  ...5000/7690
  ...6000/7690
  ...7000/7690

parsed 7690 files, 0 failures, 579,468 probe measurements

codesets present (GeneRLF):
gene_rlf
Raphaela_III_C3086      5840
Raphaela_II_C1633        962
Raphaela_I-A956          732
YL_RGB_3_C3025           108
Raphaela_II_C1633[1]      36
Iontek_1_C1209            12
Name: count, dtype: int64
metadata: /mnt/file-systems/MetadataPatients_NoPHI.xlsx  (2157 rows)
columns: ['Enrollment ID', 'Alternate Enrollment ID', 'Initials', 'Birth Date', 'Sex', 'Race', 'Ethnicity', 'Study Status', 'Diagnosis1', 'Diagnosis2', 'Notes']

initials map: 2081 auto from metadata, 42 manual overrides, 17 ambiguous (never auto-filled)

age computed for 3210/7690 files  (no DOB: 102, no collection date: 369)
collection-date source:
sample_date_source
id_field    3995
no